In [ ]:
from obstore.store import HTTPStore
from virtualizarr import open_virtual_dataset
from obspec_utils.registry import ObjectStoreRegistry
from virtual_tiff import VirtualTIFF
from urllib.parse import urlparse
from obstore.store import LocalStore

import xarray as xr
import rioxarray
import zarr
import xproj
import rasterix
from affine import Affine
from pathlib import Path
import numpy as np

%matplotlib inline
zarr.config.set({'async.concurrency': 128})


In [ ]:
# GeoTIFF RasterTypeGeoKey values
RASTER_PIXEL_IS_AREA = 1
RASTER_PIXEL_IS_POINT = 2
GEOKEY_USER_DEFINED = 32767
 
MODEL_TYPE_PROJECTED = 1
MODEL_TYPE_GEOGRAPHIC = 2
MODEL_TYPE_GEOCENTRIC = 3
 
 
def transform_from_geotiff_attrs(attrs, *, tiepoint_index=0):
    """Build a corner-referenced Affine from GeoTIFF georeferencing tags.
 
    Handles the three mechanisms the GeoTIFF spec allows, in precedence order:
      1. ModelTransformationTag  -> full 4x4, supports rotation/shear
      2. ModelPixelScaleTag + ModelTiepointTag -> axis-aligned
      3. multiple tiepoints (GCPs) -> not an affine, raises
    """
    # --- 1. Full transformation matrix (present when the grid is rotated) ---
    mt = attrs.get("model_transformation")
    if mt:
        m = list(mt)
        if len(m) != 16:
            raise ValueError(f"model_transformation must have 16 values, got {len(m)}")
        # row-major 4x4; rows 0 and 1 carry the 2-D affine
        return Affine(m[0], m[1], m[3],
                      m[4], m[5], m[7])
 
    # --- 2. Pixel scale + tiepoint ---
    scale = attrs.get("model_pixel_scale")
    tie = attrs.get("model_tiepoint")
    if not scale or not tie:
        raise ValueError("No ModelTransformation and no PixelScale/Tiepoint pair; "
                         "file is not affine-georeferenced.")
 
    if len(tie) % 6:
        raise ValueError(f"model_tiepoint length {len(tie)} is not a multiple of 6")
    n_tiepoints = len(tie) // 6
    if n_tiepoints > 1:
        # --- 3. GCPs ---
        raise ValueError(f"{n_tiepoints} tiepoints (GCPs) present; a single affine "
                         "cannot represent this. Fit with rasterio.transform.from_gcps.")
 
    off = tiepoint_index * 6
    i, j, _k, x, y, _z = tie[off:off + 6]
 
    sx, sy = float(scale[0]), float(scale[1])
    if sx <= 0 or sy <= 0:
        raise ValueError(f"ModelPixelScale must be positive, got ({sx}, {sy})")
 
    # Raster point (i, j) maps to model point (x, y). Back out pixel (0, 0).
    # +x to the right, +y up in model space => j term is added, not subtracted.
    origin_x = x - i * sx
    origin_y = y + j * sy
 
    # Tiepoint refers to the pixel CENTRE when RasterPixelIsPoint; shift to the
    # corner, since Affine/rasterio/rasterix all assume corner-referenced.
    if attrs.get("raster_type") == RASTER_PIXEL_IS_POINT:
        origin_x -= sx / 2.0
        origin_y += sy / 2.0
 
    return Affine(sx, 0.0, origin_x,
                  0.0, -sy, origin_y)

 

 
def _epsg_from_projection_geokey(proj_code):
    """ProjectionGeoKey (3074) -> EPSG, for the WGS84 UTM ranges."""
    if 16001 <= proj_code <= 16060:
        return 32600 + (proj_code - 16000)      # UTM north
    if 16101 <= proj_code <= 16160:
        return 32700 + (proj_code - 16100)      # UTM south
    raise ValueError(
        f"ProjectionGeoKey {proj_code} is not a WGS84 UTM zone; reconstruct the "
        "CRS from the geog_* keys with pyproj.CRS.from_dict / from_wkt instead."
    )
 
 
def epsg_from_geotiff_attrs(attrs):
    """Return an EPSG integer, preferring the registry code over the GeoKeys."""
    model_type = attrs.get("model_type", MODEL_TYPE_PROJECTED)
 
    if model_type == MODEL_TYPE_PROJECTED:
        code = attrs.get("projected_type")
        if code and code != GEOKEY_USER_DEFINED:
            return int(code)                     # normal case: real EPSG code
        proj_code = attrs.get("projection")
        if proj_code is None:
            raise ValueError(
                "projected_type is user-defined (32767) but no 'projection' "
                "GeoKey is present; cannot infer the CRS."
            )
        return _epsg_from_projection_geokey(int(proj_code))
 
    if model_type == MODEL_TYPE_GEOGRAPHIC:
        code = attrs.get("geographic_type")
        if code and code != GEOKEY_USER_DEFINED:
            return int(code)
        raise ValueError("User-defined geographic CRS; rebuild from geog_* keys.")
 
    raise ValueError(f"Unsupported ModelTypeGeoKey: {model_type}")
 
 
def check_hemisphere(attrs, epsg, transform):
    """Flag the HLS-style north/south mismatch. Returns (epsg, transform, notes)."""
    notes = []
    declared = attrs.get("HORIZONTAL_CS_CODE")
    if declared:
        declared_epsg = int(str(declared).split(":")[-1])
        if declared_epsg != epsg:
            notes.append(
                f"GeoKeys say EPSG:{epsg} but HORIZONTAL_CS_CODE says "
                f"EPSG:{declared_epsg}"
            )
 
    northing = transform.f                       # origin y
    is_utm_north = 32601 <= epsg <= 32660
    if is_utm_north and northing < 0:
        notes.append(
            f"origin northing {northing:,.0f} is negative under a UTM *north* CRS "
            "-> southern-hemisphere tile written with the 10,000,000 m false "
            "northing subtracted"
        )
    return notes
 
 
def normalize_southern(epsg, transform):
    """Rewrite a negative-northing UTM-north grid as its UTM-south equivalent."""
    if not (32601 <= epsg <= 32660) or transform.f >= 0:
        return epsg, transform
    south = epsg + 100                            # 326zz -> 327zz
    shifted = Affine(transform.a, transform.b, transform.c,
                     transform.d, transform.e, transform.f + 10_000_000.0)
    return south, shifted
 

In [ ]:
directory_path = Path("./input_files/")

# Find all .json files in the top-level directory
tif_files = list(directory_path.glob("*.tif"))
print("found: ", len(tif_files), " tif files")
filename = tif_files[2]
filename

In [ ]:
filepath = f"{filename.resolve().parent}/{filename.name}"

registry = ObjectStoreRegistry({"file://": LocalStore()})
parser = VirtualTIFF(ifd_layout="nested")

ms = parser(f"file://{filepath}", registry=registry)
ms.to_virtual_datatree()

# flatten out any grouped data and rename coordinates to avoid name collision

In [ ]:
ds0 = xr.open_zarr(ms, group='0', consolidated=False, zarr_format=3, decode_cf=False)
ds1 = xr.open_zarr(ms, group='1', consolidated=False, zarr_format=3, decode_cf=False)
ds2 = xr.open_zarr(ms, group='2', consolidated=False, zarr_format=3, decode_cf=False)
ds3 = xr.open_zarr(ms, group='3', consolidated=False, zarr_format=3, decode_cf=False)
ds4 = xr.open_zarr(ms, group='4', consolidated=False, zarr_format=3, decode_cf=False)

if "y" in ds1.dims:
    ds1 = ds1.rename_dims({'y':'y1', 'x':'x1'})
    ds0 = ds0.rename_dims({'y':'y0', 'x':'x0'})
if "y" in ds2.dims:
    ds2 = ds2.rename_dims({'y':'y2', 'x':'x2'})
if "y" in ds3.dims:
    ds3 = ds3.rename_dims({'y':'y3', 'x':'x3'})
if "y" in ds4.dims:
    ds4 = ds4.rename_dims({'y':'y4', 'x':'x4'})

ds = xr.merge([ds0,ds1,ds2,ds3,ds4])
ds

In [ ]:
da_vt = ds.load()
attrs = da_vt.attrs
attrs

In [ ]:
_FillValue = attrs.pop("_FillValue", None) or -9999
scale = np.float32(attrs.pop('scale_factor', None))
if scale is None:
    print('here')
    scale = attrs.pop('scale_factor', None) or 1
# scale = np.float32(scale)
print(f"Fill value= {_FillValue}, and scale factor= {scale}")

In [ ]:
scale

In [ ]:
code = attrs.get("projected_type")
if code and code != 32767:
    epsg = code                                   # this file -> 32656
else:
    epsg = _epsg_from_projection_geokey(attrs["projection"])   # previous file -> 32615
print(epsg)

In [ ]:
transform = transform_from_geotiff_attrs(attrs)
for note in check_hemisphere(attrs, epsg, transform):
    print("WARN:", note)
epsg, transform = normalize_southern(epsg, transform)   # 32656 -> 32756, y += 1e7

da_vt = da_vt.proj.assign_crs(spatial_ref=f"EPSG:{epsg}")


In [ ]:
model_pixel_scale = attrs['model_pixel_scale']
model_tiepoint = attrs['model_tiepoint']

transform = transform_from_geotiff_attrs(attrs)
try:
    index = rasterix.RasterIndex.from_transform(
        transform, width=da_vt.sizes["x0"], height=da_vt.sizes["y0"]
    )
except KeyError:
    index = rasterix.RasterIndex.from_transform(
        transform, width=da_vt.sizes["x"], height=da_vt.sizes["y"]
    )

coords = xr.Coordinates.from_xindex(index)
da_vt = da_vt.assign_coords(coords)
(da_vt.where(da_vt != _FillValue) * scale)['0'].plot.imshow(cmap="gray")